# Phân phối nhãn rủi ro theo tập dữ liệu

Notebook gán nhãn NORMAL / WARNING / HIGH cho volatility thực tế bằng các ngưỡng quantile được fit **chỉ trên target của train**, sau đó so sánh số lượng và tỷ lệ nhãn ở train, validation và test.

**Lưu ý về split của repository:** `dataset.validation` là 5 cửa sổ CV nằm bên trong `dataset.train`; các mẫu validation cũng được đưa vào lần fit cuối. Vì vậy hai cột Train và Validation có phần chồng lấp và không nên cộng số mẫu của chúng. Test là tập thời gian riêng ở cuối.

Chạy các cell lần lượt. Có thể đổi mã cổ phiếu ở cell cấu hình.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TICKER = "ACB"
CONFIG_PATH = ROOT / "config.yaml"
print(f"Mã đang phân tích: {TICKER}")


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from src.config import load_config
from src.pipeline.dataset import prepare_dataset
from src.risk.warning_engine import WarningEngine

config = load_config(CONFIG_PATH)
dataset = prepare_dataset(TICKER, config)

# Ngưỡng rủi ro được fit từ target train; áp dụng nguyên ngưỡng này cho cả ba nhóm.
warning = WarningEngine(**config["warning"]).fit(dataset.train[dataset.target])
splits = {
    "Train (final fit)": dataset.train,
    "Validation (CV, overlaps train)": dataset.validation,
    "Test": dataset.test,
}
risk_order = ["NORMAL", "WARNING", "HIGH"]
rows = []
for split_name, frame in splits.items():
    labels = warning.classify(frame[dataset.target].to_numpy())
    counts = pd.Series(labels).value_counts()
    n = len(frame)
    for level in risk_order:
        count = int(counts.get(level, 0))
        rows.append({"split": split_name, "risk_level": level, "count": count, "percent": 100 * count / n, "n": n})

distribution = pd.DataFrame(rows)
print(f"Ngưỡng train: WARNING ≥ {warning.warning_threshold:.3f}; HIGH ≥ {warning.high_threshold:.3f} điểm %")
display(distribution.pivot(index="split", columns="risk_level", values="count").reindex(columns=risk_order))
display(distribution.pivot(index="split", columns="risk_level", values="percent").reindex(columns=risk_order).round(1))


In [ ]:
colors = {"NORMAL": "#2ca25f", "WARNING": "#f0ad4e", "HIGH": "#de2d26"}
split_order = list(splits)
split_labels = [f"{name}<br>n={len(splits[name])}" for name in split_order]

# Raw counts: the hover tooltip also reports each class percentage.
fig_counts = go.Figure()
for level in risk_order:
    part = distribution[distribution["risk_level"] == level].set_index("split").loc[split_order]
    counts = part["count"].to_list()
    percents = part["percent"].to_list()
    fig_counts.add_trace(go.Bar(
        name=level, x=split_labels, y=counts, text=counts, textposition="outside",
        marker_color=colors[level], customdata=percents,
        hovertemplate="%{x}<br>" + level + ": %{y} mẫu (%{customdata:.1f}%)<extra></extra>",
    ))
fig_counts.update_layout(
    title=f"{TICKER}: số mẫu mỗi mức rủi ro — ngưỡng fit trên train",
    xaxis_title="Split dữ liệu", yaxis_title="Số mẫu",
    barmode="group", template="plotly_white", height=520,
    legend_title_text="Nhãn volatility", margin={"b": 120},
)
fig_counts.update_yaxes(rangemode="tozero")
fig_counts.show()

# Composition: percentages make the different split sizes comparable.
fig_share = go.Figure()
for level in risk_order:
    part = distribution[distribution["risk_level"] == level].set_index("split").loc[split_order]
    percents = part["percent"].to_list()
    counts = part["count"].to_list()
    fig_share.add_trace(go.Bar(
        name=level, x=split_labels, y=percents,
        text=[f"{value:.1f}%" for value in percents], textposition="inside",
        marker_color=colors[level], customdata=counts,
        hovertemplate="%{x}<br>" + level + ": %{customdata} mẫu (%{y:.1f}%)<extra></extra>",
    ))
fig_share.update_layout(
    title=f"{TICKER}: tỷ lệ nhãn trong từng split",
    xaxis_title="Split dữ liệu", yaxis_title="Tỷ lệ",
    barmode="stack", template="plotly_white", height=520,
    legend_title_text="Nhãn volatility", margin={"b": 120},
)
fig_share.update_yaxes(range=[0, 100], ticksuffix="%")
fig_share.show()


## Cách đọc kết quả

Tập train được chia ngưỡng quantile nên tỷ lệ HIGH tại train thường xấp xỉ phần đuôi do `q_high` quy định. Validation hoặc test có thể lệch đáng kể vì phân phối volatility thay đổi theo thời gian. So sánh cả **số lượng** lẫn **tỷ lệ**: số mẫu tuyệt đối phụ thuộc kích thước split, còn tỷ lệ cho thấy mức mất cân bằng nhãn rõ hơn.